# 02 — Train

Runs `python -m ssm_phylo.train --config <picked> --resume latest` with streaming output, then plots the loss curve from `metrics.csv`.

> **WARNING: Colab sessions can die at any moment.** Training is checkpoint-resumable: if the session dies, just re-run this notebook — `--resume latest` pulls the Drive mirror and continues from the last saved step (never double-counts steps).

In [ ]:
import os, subprocess, sys, tempfile, pathlib
REPO_DIR = os.path.abspath(os.getcwd())
sys.path.insert(0, REPO_DIR)
DRIVE_SKIP = os.environ.get("COLAB_DRIVE_SKIP", "0") == "1"
if not os.environ.get("COLAB_DRIVE"):
    COLAB_DRIVE = ("/content/drive/MyDrive/ssm-phylo" if not DRIVE_SKIP
                   else tempfile.mkdtemp(prefix="ssm_drive_"))
    os.environ.update(
        COLAB_DRIVE=COLAB_DRIVE,
        DATA_DIR=f"{COLAB_DRIVE}/data",
        LOCAL_CKPT_DIR="/content/ckpts" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_ckpts",
        LOCAL_DATA_DIR="/content/data" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_data",
        CKPT_DIR=f"{COLAB_DRIVE}/checkpoints",
        RESULTS_DIR=f"{COLAB_DRIVE}/results",
    )
    for d in [os.environ["LOCAL_DATA_DIR"], os.environ["LOCAL_CKPT_DIR"],
              os.environ["DATA_DIR"], os.environ["CKPT_DIR"], os.environ["RESULTS_DIR"]]:
        pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", os.environ.get("DATA_DIR"))
print("CKPT_DIR:", os.environ.get("CKPT_DIR"))
print("RESULTS_DIR:", os.environ.get("RESULTS_DIR"))


In [ ]:
#@title Pull latest checkpoints from Drive (resume safety, skipped on fresh sessions)
if not DRIVE_SKIP and os.environ.get("CKPT_DIR") and os.environ.get("LOCAL_CKPT_DIR"):
    pull = subprocess.run(["bash", f"{REPO_DIR}/scripts/sync_drive.sh", "pull"],
                          env=os.environ, text=True)
    print("pull exit:", pull.returncode)
else:
    print("skipped (no Drive or COLAB_DRIVE_SKIP=1)")

In [ ]:
#@title Training configuration
config = "train_small"  #@param ["train_small", "train_l4"] {type:"raw"}
extra_args = os.environ.get("SSM_PHYLO_NOTEBOOK_TRAIN_EXTRA", "")  # e.g. "--max-steps 2000"
cmd = [sys.executable, "-m", "ssm_phylo.train",
       "--config", f"{REPO_DIR}/configs/{config}.yaml",
       "--resume", "latest",
       "--data-dir", os.environ["DATA_DIR"],
       "--ckpt-dir", os.environ["CKPT_DIR"]]
if os.environ.get("RESULTS_DIR"):
    cmd += ["--results-dir", os.environ["RESULTS_DIR"]]
cmd += extra_args.split()
print(" ".join(cmd))

In [ ]:
#@title Train (streaming output; Ctrl-C / session death is safe — rerun to resume)
proc = subprocess.Popen(cmd, env=os.environ, cwd=REPO_DIR, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
print("\ntrain exit:", rc)

In [ ]:
#@title Loss curve from metrics.csv
import csv
import matplotlib.pyplot as plt

local_data = os.environ.get("LOCAL_DATA_DIR", "/content/data")
csv_path = os.path.join(local_data, "metrics.csv")
if not os.path.exists(csv_path):
    print(f"no metrics.csv at {csv_path} — train at least once")
else:
    rows = [r for r in csv.DictReader(open(csv_path)) if r.get("loss")]
    steps = [int(r["step"]) for r in rows]
    plt.figure(figsize=(6, 3))
    plt.plot(steps, [float(r["loss"]) for r in rows], label="loss")
    plt.plot(steps, [float(r["mae"]) for r in rows], label="mae (raw)")
    plt.xlabel("global step"); plt.ylabel("loss")
    plt.legend(); plt.title(f"training curve — {len(rows)} logged steps")
    plt.show()